# Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

In [2]:
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)

In [3]:
app_train = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/application_train.csv')
app_test = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/application_test.csv')
bureau = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/bureau.csv')
bureau_balance = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv')
pos_cash_balance = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv')
previous_app = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/previous_application.csv')
installments_payments = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv')
credit_card_balance = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv')

# Feature enginering

On applique les mêmes transformations au jeu d’entraînement et au jeu test afin d’assurer la cohérence des données utilisées par les modèles.

## app_train

**Ratios financiers**

Dans cette première étape de feature engineering, on construit des ratios financiers à partir des variables brutes du dataset (*app_train* et *app_test*).

Les montants absolus comme le crédit, le revenu ou l’annuité ne sont pas toujours suffisants pour évaluer le risque.
Ce qui est réellement informatif, c’est la proportion entre ces montants.

Par exemple :
- Un crédit élevé n’est pas forcément risqué si le revenu est également élevé.
- Une annuité importante peut être problématique si elle représente une grande part du revenu mensuel.

Ces ratios sont largement utilisés en scoring bancaire car ils permettent de mesurer la pression financière réelle supportée par le client.

In [4]:
for df in [app_train, app_test]:
    
    # Ce ratio mesure l’importance du crédit demandé par rapport au revenu total.
    # Un client qui emprunte un montant très élevé par rapport à son revenu présente généralement un risque plus important.
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)
    
    # Ce ratio permet d’évaluer le poids des remboursements mensuels par rapport au revenu.
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
    
    # Cette variable donne une approximation de la durée du crédit.
    # Elle permet de distinguer des crédits courts et des crédits longs, ce qui peut influencer le risque de défaut.
    df['CREDIT_TERM'] = df['AMT_ANNUITY'] / (df['AMT_CREDIT'] + 1)
    
    # Ce ratio compare le prix du bien au montant financé.
    # Il permet d’identifier si le client finance la totalité du bien ou s’il dispose d’un apport personnel.
    df['GOODS_PRICE_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + 1)

print(app_train.iloc[:, [0] + list(range(-4, 0))].head(5))

   SK_ID_CURR  CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO  CREDIT_TERM  \
0      100002             2.007879              0.121977     0.060749   
1      100003             4.790732              0.132216     0.027598   
2      100004             1.999970              0.099999     0.050000   
3      100006             2.316150              0.219898     0.094941   
4      100007             4.222187              0.179961     0.042623   

   GOODS_PRICE_CREDIT_RATIO  
0                  0.863259  
1                  0.873210  
2                  0.999993  
3                  0.949842  
4                  0.999998  


**Variables démographiques ajustées**

Dans cette partie, on enrichit les variables liées au profil du client.

L’objectif est de contextualiser les informations disponibles :
- La stabilité professionnelle doit être analysée en fonction de l’âge.
- Le revenu doit être interprété en tenant compte de la taille du foyer.
- La présence d’enfants peut influencer indirectement la capacité de remboursement.

Ces transformations permettent de créer des indicateurs plus pertinents que les variables brutes, en intégrant une dimension socio-économique au modèle.

In [5]:
for df in [app_train, app_test]:
    
    # Cette variable met en relation la durée d’emploi et l’âge.
    # Elle permet d’obtenir un indicateur de stabilité professionnelle en tenant compte du cycle de vie du client.
    df['DAYS_EMPLOYED_RATIO'] = df['DAYS_EMPLOYED'] / (df['DAYS_BIRTH'] + 1)
    
    # Ici, le revenu total est ajusté par la taille du foyer.
    # Un revenu élevé n’a pas le même impact selon que le client vit seul ou avec plusieurs personnes à charge.
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + 1)
    
    # Cette variable représente la proportion d’enfants dans le foyer.
    # Elle peut influencer indirectement la capacité de remboursement.
    df['CHILDREN_RATIO'] = df['CNT_CHILDREN'] / (df['CNT_FAM_MEMBERS'] + 1)

print(app_train.iloc[:, [0] + list(range(-3, 0))].head(5))

   SK_ID_CURR  DAYS_EMPLOYED_RATIO  INCOME_PER_PERSON  CHILDREN_RATIO
0      100002             0.067336           101250.0             0.0
1      100003             0.070866            90000.0             0.0
2      100004             0.011814            33750.0             0.0
3      100006             0.159914            45000.0             0.0
4      100007             0.152426            60750.0             0.0


**Conversion des variables temporelles**

Les variables `DAYS_BIRTH` et `DAYS_EMPLOYED` sont exprimées en jours négatifs dans le dataset.

Afin d’améliorer l’interprétation et de rendre ces variables plus cohérentes d’un point de vue statistique, on les convertit en années.

Cette transformation permet :
- Une meilleure lisibilité des variables.
- Une échelle plus adaptée aux modèles linéaires.
- Une réduction des effets liés aux très grandes valeurs numériques.

In [6]:
for df in [app_train, app_test]:
    
    df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365
    df['EMPLOYED_YEARS'] = -df['DAYS_EMPLOYED'] / 365

print(app_train.iloc[:, [0] + list(range(-2, 0))].head(5))

   SK_ID_CURR  AGE_YEARS  EMPLOYED_YEARS
0      100002  25.920548        1.745205
1      100003  45.931507        3.254795
2      100004  52.180822        0.616438
3      100006  52.068493        8.326027
4      100007  54.608219        8.323288


**Création d’interactions non-linéaires**

Les modèles linéaires, comme la régression logistique, ne capturent pas naturellement les interactions complexes entre variables.

Pour compenser cette limite, on crée manuellement des interactions entre :
- Le revenu
- Le montant du crédit
- L’âge

L’objectif est de modéliser des relations plus complexes, par exemple, un crédit élevé peut être moins risqué si le revenu est important ou si le client est plus âgé.

Cette étape permet d’introduire une dimension non-linéaire tout en conservant un modèle linéaire.

In [7]:
for df in [app_train, app_test]:
    
    df['INCOME_CREDIT_AGE'] = df['AMT_INCOME_TOTAL'] * df['AMT_CREDIT'] / (df['AGE_YEARS'] + 1)

print(app_train.iloc[:, [0] + list(range(-1, 0))].head(5))

   SK_ID_CURR  INCOME_CREDIT_AGE
0      100002       3.058481e+09
1      100003       7.441604e+09
2      100004       1.713494e+08
3      100006       7.954275e+08
4      100007       1.120868e+09


**Agrégation des scores externes**

Les variables `EXT_SOURCE_1`, `EXT_SOURCE_2` et `EXT_SOURCE_3` sont particulièrement importantes dans ce dataset.

Elles représentent des scores externes liés au risque du client et sont connues pour être fortement prédictives.

Plutôt que d’utiliser ces variables séparément, on crée plusieurs statistiques :
- Moyenne : estimation globale du niveau de risque.
- Maximum et minimum : identification du meilleur et du pire score.
- Écart-type : mesure de la cohérence entre les scores.
- Produit : effet multiplicatif, pénalisant fortement un score très faible.

Cette agrégation permet d’extraire davantage d’information à partir de ces variables clés.

In [8]:
for df in [app_train, app_test]:
    
    # Moyenne des scores externes, qui donne une estimation globale.
    df['EXT_SOURCE_MEAN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
    
    # Valeur maximale, correspondant au meilleur score.
    df['EXT_SOURCE_MAX'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].max(axis=1)
    
    # Valeur minimale, correspondant au score le plus défavorable.
    df['EXT_SOURCE_MIN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].min(axis=1)
    
    # Écart-type des scores, qui mesure leur cohérence.
    df['EXT_SOURCE_STD'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].std(axis=1)
    
    # Produit des scores, qui amplifie les cas où tous les scores sont élevés ou pénalise fortement lorsqu’un score est très faible.
    df['EXT_SOURCE_PROD'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']

print(app_train.iloc[:, [0] + list(range(-5, 0))].head(5))

   SK_ID_CURR  EXT_SOURCE_MEAN  EXT_SOURCE_MAX  EXT_SOURCE_MIN  \
0      100002         0.161787        0.262949        0.083037   
1      100003         0.466757        0.622246        0.311267   
2      100004         0.642739        0.729567        0.555912   
3      100006         0.650442        0.650442        0.650442   
4      100007         0.322738        0.322738        0.322738   

   EXT_SOURCE_STD  EXT_SOURCE_PROD  
0        0.092026         0.003043  
1        0.219895              NaN  
2        0.122792              NaN  
3             NaN              NaN  
4             NaN              NaN  


**Transformation en variables binaires**

Certaines relations entre variables et risque peuvent présenter des effets de seuil.

Par exemple :
- Un revenu très élevé peut réduire significativement le risque.
- Une annuité trop importante peut augmenter fortement la probabilité de défaut.

Afin de permettre aux modèles linéaires de capturer ces effets de seuil, on transforme certaines variables continues en indicateurs binaires basés sur la médiane.

Cette approche simplifie l’information tout en conservant une capacité discriminante.

In [9]:
for df in [app_train, app_test]:
    
    # Indique si l’annuité est supérieure à la médiane.
    df['ANNUITY_HIGH'] = (df['AMT_ANNUITY'] > df['AMT_ANNUITY'].median()).astype(int)
    
    # Indique si le revenu est supérieur à la médiane.
    df['INCOME_HIGH'] = (df['AMT_INCOME_TOTAL'] > df['AMT_INCOME_TOTAL'].median()).astype(int)

print(app_train.iloc[:, [0] + list(range(-2, 0))].head(5))

   SK_ID_CURR  ANNUITY_HIGH  INCOME_HIGH
0      100002             0            1
1      100003             1            1
2      100004             0            0
3      100006             1            0
4      100007             0            0


## bureau

Le fichier *bureau* contient l’historique des crédits contractés par les clients auprès d’autres institutions financières.

Chaque client (`SK_ID_CURR`) peut apparaître plusieurs fois dans cette table, car il peut avoir contracté plusieurs crédits (`SK_ID_BUREAU`).

L’objectif de cette étape est de transformer ces données transactionnelles (en plusieurs lignes par client) en variables agrégées exploitables par un modèle.

On réalise donc une agrégation par client afin de résumer :
- Le nombre total de crédits contractés
- Le comportement de remboursement
- Le niveau d’endettement global
- Les montants en retard
- La diversité des types de crédits

Ces variables sont particulièrement importantes en scoring bancaire, car l’historique de crédit est souvent l’un des meilleurs prédicteurs du défaut.

In [10]:
bureau_agg = bureau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU': 'count',
    'DAYS_CREDIT': ['min', 'max', 'mean', 'std'],
    'CREDIT_DAY_OVERDUE': ['max', 'mean', 'sum'],
    'DAYS_CREDIT_ENDDATE': ['min', 'max', 'mean'],
    'DAYS_ENDDATE_FACT': ['min', 'max', 'mean'],
    'AMT_CREDIT_MAX_OVERDUE': ['max', 'mean'],
    'AMT_CREDIT_SUM': ['sum', 'mean', 'max', 'std'],
    'AMT_CREDIT_SUM_DEBT': ['sum', 'mean', 'max'],
    'AMT_CREDIT_SUM_LIMIT': ['sum', 'mean', 'max'],
    'AMT_CREDIT_SUM_OVERDUE': ['sum', 'mean', 'max'],
    'CREDIT_TYPE': 'nunique',
    'DAYS_CREDIT_UPDATE': ['min', 'max', 'mean'],
    'AMT_ANNUITY': ['mean', 'max', 'sum']
})

bureau_agg.columns = ['BUREAU_' + '_'.join(col).strip() for col in bureau_agg.columns.values]
bureau_agg.reset_index(inplace=True)

bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] = bureau_agg['BUREAU_AMT_CREDIT_SUM_DEBT_sum'] / (bureau_agg['BUREAU_AMT_CREDIT_SUM_sum'] + 1)
bureau_agg['BUREAU_OVERDUE_RATIO'] = bureau_agg['BUREAU_AMT_CREDIT_SUM_OVERDUE_sum'] / (bureau_agg['BUREAU_AMT_CREDIT_SUM_sum'] + 1)

app_train = app_train.merge(bureau_agg, on='SK_ID_CURR', how='left')
app_test = app_test.merge(bureau_agg, on='SK_ID_CURR', how='left')

print(f"Shape après bureau: {app_train.shape}")

Shape après bureau: (307511, 177)


## bureau_balance

La table *bureau_balance* contient l’historique mensuel des crédits déclarés auprès d’autres institutions financières. 

Chaque crédit (`SK_ID_BUREAU`) apparaît plusieurs fois, car son statut est suivi mois par mois.

L’objectif est de transformer cet historique en variables agrégées exploitables par le modèle.

On commence par agréger les données au niveau du crédit (`SK_ID_BUREAU`) afin de résumer :
- Le nombre de fois où le crédit est clôturé (STATUS = 'C')
- La durée d’observation (mois minimum et maximum)
- Le nombre total de mois enregistrés
- Le mois moyen observé

Ces informations sont ensuite fusionnées avec la table *bureau*, puis une seconde agrégation est réalisée au niveau du client (`SK_ID_CURR`) afin d’obtenir une vision globale de son historique de crédit externe.

On calcule ainsi des indicateurs comme :
- Le nombre total et moyen de crédits clôturés
- La durée globale d’historique
- Le nombre total de mois déclarés
- La moyenne des durées observées
- 
Enfin, ces nouvelles variables (préfixées par BB_) sont fusionnées avec *app_train* et *app_test*, ce qui permet d’enrichir le dataset avec des informations supplémentaires sur l’historique bancaire externe des clients.

In [11]:
bureau_balance_agg = bureau_balance.groupby('SK_ID_BUREAU').agg({
    'STATUS': lambda x: (x == 'C').sum(),
    'MONTHS_BALANCE': ['min', 'max', 'size', 'mean']
})

bureau_balance_agg.columns = ['BB_STATUS_CLOSED', 'BB_MONTHS_MIN', 'BB_MONTHS_MAX', 'BB_COUNT', 'BB_MONTHS_MEAN']
bureau_balance_agg.reset_index(inplace=True)

bureau_with_balance = bureau.merge(bureau_balance_agg, on='SK_ID_BUREAU', how='left')
bureau_balance_agg2 = bureau_with_balance.groupby('SK_ID_CURR').agg({
    'BB_STATUS_CLOSED': ['sum', 'mean', 'max'],
    'BB_MONTHS_MIN': 'min',
    'BB_MONTHS_MAX': 'max',
    'BB_COUNT': ['sum', 'mean', 'max'],
    'BB_MONTHS_MEAN': 'mean'
})

bureau_balance_agg2.columns = ['BB_' + '_'.join(col).strip() for col in bureau_balance_agg2.columns.values]
bureau_balance_agg2.reset_index(inplace=True)

app_train = app_train.merge(bureau_balance_agg2, on='SK_ID_CURR', how='left')
app_test = app_test.merge(bureau_balance_agg2, on='SK_ID_CURR', how='left')

print(f"Shape après bureau_balance: {app_train.shape}")

Shape après bureau_balance: (307511, 186)


## previous_application

L'idée ici, c'est de ne pas juste regarder le profil actuel du client, mais de voir comment il s'est comporté par le passé. On résume toutes ses anciennes demandes de prêt en quelques indicateurs clés pour que notre modèle comprenne mieux son profil.

On regarde les statuts, on crée des colonnes pour savoir si ses anciens dossiers ont été Acceptés, Refusés ou Annulés.

On fait de l'agrégation avec **groupby**. Par exemple un client peut avoir eu 10 prêts avant, donc on regroupe tout par `SK_ID_CURR` (l'ID du client) et on calcule :
- Le nombre total d'anciennes demandes.
- Le minimum, le maximum, la moyenne et la somme de ce qu'il a demandé (crédit, annuités, prix des biens).
- On regarde quand les décisions ont été prises et quand les remboursements ont eu lieu (`DAYS_DECISION`, `DAYS_TERMINATION`).

On calcule nos propres scores :
- Est-ce qu'on lui dit souvent "oui" pour un crédit?
- Est-ce qu'il se fait souvent refuser un crédit ?

On colle toutes ces nouvelles infos (qui commencent par PREV_) dans nos tableaux principaux *app_train* et *app_test*.

In [12]:
for df in [previous_app]:
    df['APP_APPROVED'] = (df['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
    df['APP_REFUSED'] = (df['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)
    df['APP_CANCELED'] = (df['NAME_CONTRACT_STATUS'] == 'Canceled').astype(int)

prev_agg = previous_app.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'AMT_ANNUITY': ['min', 'max', 'mean', 'sum'],
    'AMT_APPLICATION': ['min', 'max', 'mean', 'sum'],
    'AMT_CREDIT': ['min', 'max', 'mean', 'sum'],
    'AMT_DOWN_PAYMENT': ['min', 'max', 'mean', 'sum'],
    'AMT_GOODS_PRICE': ['min', 'max', 'mean', 'sum'],
    'RATE_DOWN_PAYMENT': ['min', 'max', 'mean'],
    'RATE_INTEREST_PRIMARY': ['min', 'max', 'mean'],
    'RATE_INTEREST_PRIVILEGED': ['min', 'max', 'mean'],
    'DAYS_DECISION': ['min', 'max', 'mean'],
    'DAYS_FIRST_DRAWING': ['min', 'max', 'mean'],
    'DAYS_FIRST_DUE': ['min', 'max', 'mean'],
    'DAYS_LAST_DUE': ['min', 'max', 'mean'],
    'DAYS_TERMINATION': ['min', 'max', 'mean'],
    'CNT_PAYMENT': ['min', 'max', 'mean', 'sum'],
    'APP_APPROVED': 'sum',
    'APP_REFUSED': 'sum',
    'APP_CANCELED': 'sum'
})

prev_agg.columns = ['PREV_' + '_'.join(col).strip() for col in prev_agg.columns.values]
prev_agg.reset_index(inplace=True)

prev_agg['PREV_APPROVAL_RATE'] = prev_agg['PREV_APP_APPROVED_sum'] / (prev_agg['PREV_SK_ID_PREV_count'] + 1)
prev_agg['PREV_REFUSED_RATE'] = prev_agg['PREV_APP_REFUSED_sum'] / (prev_agg['PREV_SK_ID_PREV_count'] + 1)

app_train = app_train.merge(prev_agg, on='SK_ID_CURR', how='left')
app_test = app_test.merge(prev_agg, on='SK_ID_CURR', how='left')

print(f"Shape après previous_app: {app_train.shape}")

Shape après previous_app: (307511, 240)


## pos_cash_balance

La table *POS_CASH_balance* contient l’historique mensuel des crédits en cours des clients.

Chaque client (`SK_ID_CURR`) apparaît plusieurs fois, car chaque ligne correspond à un mois d’activité.

L’objectif est de transformer ces données mensuelles en variables agrégées exploitables par le modèle.

On réalise donc une agrégation par client afin de résumer :
- Le nombre de mois d’historique disponible
- Le montant moyen des mensualités et le capital restant dû
- Le niveau de retard de paiement (retard moyen et maximal)

Un ratio de retard (`POS_DPD_RATIO`) est également créé pour mesurer la fréquence des retards sur toute la période observée.

Ces nouvelles variables (préfixées par POS_) sont ensuite fusionnées avec *app_train* et *app_test* afin d’enrichir les informations disponibles sur chaque client.

In [13]:
pos_agg = pos_cash_balance.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'MONTHS_BALANCE': ['min', 'max', 'mean', 'size'],
    'CNT_INSTALMENT': ['min', 'max', 'mean', 'sum'],
    'CNT_INSTALMENT_FUTURE': ['min', 'max', 'mean', 'sum'],
    'SK_DPD': ['max', 'mean', 'sum'],
    'SK_DPD_DEF': ['max', 'mean', 'sum']
})
pos_agg.columns = ['POS_' + '_'.join(col).strip() for col in pos_agg.columns.values]
pos_agg.reset_index(inplace=True)

# Features additionnelles
pos_agg['POS_DPD_RATIO'] = pos_agg['POS_SK_DPD_sum'] / (pos_agg['POS_MONTHS_BALANCE_size'] + 1)

app_train = app_train.merge(pos_agg, on='SK_ID_CURR', how='left')
app_test = app_test.merge(pos_agg, on='SK_ID_CURR', how='left')

print(f"Shape après pos_cash_balance: {app_train.shape}")

Shape après pos_cash_balance: (307511, 260)


## installments_payments

La table *installments_payments* permet d’analyser si les clients respectent leurs engagements de remboursement.

Chaque ligne correspond à une échéance de paiement, ce qui implique plusieurs lignes par client (`SK_ID_CURR`).

L’objectif est de détecter les comportements à risque, comme les retards ou les paiements incomplets.

On commence par créer des indicateurs élémentaires :
- L’écart entre le montant payé et le montant dû
- L’écart entre la date réelle et la date prévue
- Des variables binaires signalant les paiements en retard, en avance ou incomplets

Ces informations sont ensuite agrégées par client pour résumer :
- La régularité des paiements
- La solidité financière (montant total payé vs dû)
- La stabilité du comportement (écart-type)

Des ratios synthétiques, comme le taux de retard et le taux de sous-paiement, sont également calculés.

Les variables obtenues (préfixe INST_) sont enfin fusionnées avec *app_train* et *app_test*.

In [14]:
installments_payments['PAYMENT_DIFF'] = installments_payments['AMT_PAYMENT'] - installments_payments['AMT_INSTALMENT']
installments_payments['PAYMENT_RATIO'] = installments_payments['AMT_PAYMENT'] / (installments_payments['AMT_INSTALMENT'] + 1)
installments_payments['DAYS_DIFF'] = installments_payments['DAYS_ENTRY_PAYMENT'] - installments_payments['DAYS_INSTALMENT']
installments_payments['LATE_PAYMENT'] = (installments_payments['DAYS_DIFF'] > 0).astype(int)
installments_payments['EARLY_PAYMENT'] = (installments_payments['DAYS_DIFF'] < -5).astype(int)
installments_payments['UNDERPAID'] = (installments_payments['PAYMENT_DIFF'] < 0).astype(int)

inst_agg = installments_payments.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'NUM_INSTALMENT_NUMBER': ['max', 'mean'],
    'DAYS_INSTALMENT': ['min', 'max', 'mean'],
    'DAYS_ENTRY_PAYMENT': ['min', 'max', 'mean'],
    'AMT_INSTALMENT': ['min', 'max', 'mean', 'sum'],
    'AMT_PAYMENT': ['min', 'max', 'mean', 'sum'],
    'PAYMENT_DIFF': ['min', 'max', 'mean', 'sum', 'std'],
    'PAYMENT_RATIO': ['min', 'max', 'mean', 'std'],
    'DAYS_DIFF': ['min', 'max', 'mean', 'std'],
    'LATE_PAYMENT': ['sum', 'mean', 'max'],
    'EARLY_PAYMENT': ['sum', 'mean'],
    'UNDERPAID': ['sum', 'mean']
})

inst_agg.columns = ['INST_' + '_'.join(col).strip() for col in inst_agg.columns.values]
inst_agg.reset_index(inplace=True)

inst_agg['INST_LATE_PAYMENT_RATIO'] = inst_agg['INST_LATE_PAYMENT_sum'] / (inst_agg['INST_SK_ID_PREV_count'] + 1)
inst_agg['INST_UNDERPAID_RATIO'] = inst_agg['INST_UNDERPAID_sum'] / (inst_agg['INST_SK_ID_PREV_count'] + 1)

app_train = app_train.merge(inst_agg, on='SK_ID_CURR', how='left')
app_test = app_test.merge(inst_agg, on='SK_ID_CURR', how='left')

print(f"Shape après installments_payments: {app_train.shape}")

Shape après installments_payments: (307511, 299)


## credit_card_balance

La table *credit_card_balance* regroupe l’historique d’utilisation des cartes de crédit des clients.

Chaque client (`SK_ID_CURR`) possède plusieurs lignes correspondant aux différents mois d’activité.

L’objectif est de résumer l’utilisation réelle du crédit afin d’identifier les profils potentiellement risqués.

On agrège les données par client pour analyser :
- Le niveau du solde (moyen et maximal)
- L’utilisation de la limite de crédit
- Les habitudes de retrait et de dépense
- Les remboursements effectués
- Les retards de paiement

À partir de ces informations, on crée plusieurs ratios clés :
- Le taux d’utilisation de la carte
- Le ratio des dépenses par rapport à la limite
- La fréquence des retards

Ces nouvelles variables (préfixe CC_) sont ensuite fusionnées avec *app_train* et *app_test*. La dimension finale du dataset est affichée afin de connaître le nombre total de variables utilisées pour entraîner le modèle.

In [15]:
cc_agg = credit_card_balance.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count',
    'MONTHS_BALANCE': ['min', 'max', 'mean', 'size'],
    'AMT_BALANCE': ['min', 'max', 'mean', 'sum'],
    'AMT_CREDIT_LIMIT_ACTUAL': ['min', 'max', 'mean'],
    'AMT_DRAWINGS_ATM_CURRENT': ['min', 'max', 'mean', 'sum'],
    'AMT_DRAWINGS_CURRENT': ['min', 'max', 'mean', 'sum'],
    'AMT_DRAWINGS_POS_CURRENT': ['min', 'max', 'mean', 'sum'],
    'AMT_INST_MIN_REGULARITY': ['min', 'max', 'mean'],
    'AMT_PAYMENT_CURRENT': ['min', 'max', 'mean', 'sum'],
    'AMT_PAYMENT_TOTAL_CURRENT': ['min', 'max', 'mean', 'sum'],
    'AMT_RECEIVABLE_PRINCIPAL': ['min', 'max', 'mean', 'sum'],
    'AMT_RECIVABLE': ['min', 'max', 'mean', 'sum'],
    'AMT_TOTAL_RECEIVABLE': ['min', 'max', 'mean', 'sum'],
    'CNT_DRAWINGS_ATM_CURRENT': ['min', 'max', 'mean', 'sum'],
    'CNT_DRAWINGS_CURRENT': ['min', 'max', 'mean', 'sum'],
    'CNT_DRAWINGS_POS_CURRENT': ['min', 'max', 'mean', 'sum'],
    'SK_DPD': ['max', 'mean', 'sum'],
    'SK_DPD_DEF': ['max', 'mean', 'sum']
})

cc_agg.columns = ['CC_' + '_'.join(col).strip() for col in cc_agg.columns.values]
cc_agg.reset_index(inplace=True)

cc_agg['CC_BALANCE_LIMIT_RATIO'] = cc_agg['CC_AMT_BALANCE_mean'] / (cc_agg['CC_AMT_CREDIT_LIMIT_ACTUAL_mean'] + 1)
cc_agg['CC_DRAWINGS_RATIO'] = cc_agg['CC_AMT_DRAWINGS_CURRENT_sum'] / (cc_agg['CC_AMT_CREDIT_LIMIT_ACTUAL_mean'] + 1)
cc_agg['CC_DPD_RATIO'] = cc_agg['CC_SK_DPD_sum'] / (cc_agg['CC_MONTHS_BALANCE_size'] + 1)

app_train = app_train.merge(cc_agg, on='SK_ID_CURR', how='left')
app_test = app_test.merge(cc_agg, on='SK_ID_CURR', how='left')

print(f"Shape finale: {app_train.shape}")

Shape finale: (307511, 367)


# Gestion des valeurs manquantes

On commence par calculer le nombre de valeurs manquantes pour chaque colonne du dataset *app_train*, puis on transforme ce nombre en pourcentage.

On construit ensuite un dataframe *missing_df* récapitulatif contenant :
- Le nom de la variable (`Column`)
- Son pourcentage de valeurs manquantes (`Missing_Pct`)


On ne garde que les colonnes qui ont au moins une valeur manquante et on les trie par ordre décroissant afin d’identifier facilement les variables les plus problématiques.


In [16]:
missing_train = app_train.isnull().sum()
missing_train_pct = (missing_train / len(app_train)) * 100
missing_df = pd.DataFrame({
    'Colonnes': missing_train.index,
    'Valeurs Manquantes': missing_train_pct.values
})
missing_df = missing_df[missing_df['Valeurs Manquantes'] > 0].sort_values('Valeurs Manquantes', ascending=False)

missing_df.head(10)

,Colonnes,Valeurs Manquantes
211,PREV_RATE_INTEREST_PRIMARY_max,98.501192
210,PREV_RATE_INTEREST_PRIMARY_min,98.501192
213,PREV_RATE_INTEREST_PRIVILEGED_min,98.501192
212,PREV_RATE_INTEREST_PRIMARY_mean,98.501192
215,PREV_RATE_INTEREST_PRIVILEGED_mean,98.501192
214,PREV_RATE_INTEREST_PRIVILEGED_max,98.501192
326,CC_AMT_PAYMENT_CURRENT_min,80.143800
327,CC_AMT_PAYMENT_CURRENT_max,80.143800
328,CC_AMT_PAYMENT_CURRENT_mean,80.143800
321,CC_AMT_DRAWINGS_POS_CURRENT_mean,80.117784


On fixe un seuil de **70 %** de valeurs manquantes. Toutes les colonnes dépassant ce seuil sont sélectionnées pour être supprimées.

Ce choix est volontairement conservateur car au-delà de ce seuil, la variable contient trop peu d’information exploitable.

On retire explicitement `TARGET` et `SK_ID_CURR` de cette liste car :
- TARGET est la variable cible
- SK_ID_CURR est l’identifiant des individus

In [17]:
threshold = 70 
cols_to_drop = missing_df[missing_df['Valeurs Manquantes'] > threshold]['Colonnes'].tolist()
cols_to_drop = [col for col in cols_to_drop if col not in ['TARGET', 'SK_ID_CURR']]

print(f"Nombre de colonnes supprimées: {len(cols_to_drop)}")

Nombre de colonnes supprimées: 83


On retire les colonnes à la fois du jeu d’entraînement *app_train* et du jeu de test *app_test* afin de garder exactement les mêmes variables dans les deux datasets.

In [18]:
if len(cols_to_drop) > 0:
    app_train = app_train.drop(columns=cols_to_drop)
    app_test = app_test.drop(columns=cols_to_drop)

print(f"Taille de notre dataframe app_train: {app_train.shape}")

Taille de notre dataframe app_train: (307511, 284)


# Preprocessing

On sélectionne toutes les colonnes de type *object*, c’est-à-dire les variables catégorielles. Ces variables ne peuvent pas être utilisées directement par les modèles de machine learning, il faut donc les encoder.

In [19]:
categorical_cols = app_train.select_dtypes(include=['object']).columns.tolist()

print(f"Nombre de colonnes catégorielles: {len(categorical_cols)}")

Nombre de colonnes catégorielles: 16


On utilise un **LabelEncoder** pour transformer les catégories en valeurs numériques.

Pour éviter toute incohérence entre le train et le test, on concatène les deux colonnes avant d’ajuster l’encodeur. Cela permet de garantir que chaque catégorie reçoit le même encodage dans les deux jeux de données.

Chaque modalité est alors remplacée par un entier.

In [20]:
le = LabelEncoder()
for col in categorical_cols:
    if col in app_train.columns:
        combined = pd.concat([app_train[col], app_test[col]]).astype(str)
        le.fit(combined)
        app_train[col] = le.transform(app_train[col].astype(str))
        app_test[col] = le.transform(app_test[col].astype(str))

On sépare :
- X : les variables explicatives
- y : la variable cible (`TARGET`)
- X_test_final : les variables explicatives du jeu de test

La colonne `SK_ID_CURR` est supprimée car c’est un identifiant et elle n’apporte aucune information prédictive.

In [21]:
X = app_train.drop(['TARGET', 'SK_ID_CURR'], axis=1)
y = app_train['TARGET']
X_test_final = app_test.drop(['SK_ID_CURR'], axis=1)

On commence par sélectionner les colonnes numériques, puis on utilise **np.isinf()** pour vérifier si elles contiennent des inf ou -inf. La méthode **any().any()** permet de savoir rapidement s’il y a au moins une valeur infinie dans le dataset.

In [22]:
num_cols = X.select_dtypes(include=[np.number])
np.isinf(num_cols).any().any()   

np.False_

Une imputation par la médiane est ensuite appliquée pour remplacer l’ensemble des valeurs manquantes. La médiane est choisie car elle est plus robuste aux valeurs extrêmes que la moyenne.

L’imputer est ajusté uniquement sur les données d’entraînement (**fit_transform**), puis appliqué aux données de test (**transform**) afin d’éviter toute fuite d’information.

Cette procédure garantit que le dataset final ne contient plus aucune valeur invalide et peut être utilisé pour l’entraînement du modèle.


In [23]:
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test_final), columns=X_test_final.columns)


On divise les données en :
- 80 % pour l’entraînement
- 20 % pour la validation

On utilise un split stratifié (**stratify=y**) pour conserver la même proportion de classes dans les deux ensembles.

In [24]:
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

On applique une standardisation des variables :
- centrage (moyenne = 0)
- réduction (écart-type = 1)

Le scaler est ajusté uniquement sur les données d’entraînement, puis appliqué aux données de validation et de test pour éviter toute fuite d’information.

In [25]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_split)
X_val_scaled = scaler.transform(X_val_split)
X_test_scaled = scaler.transform(X_test_imputed)

# Régression logistique

Pour entraîner le premier modèle, on réalise une régression logistique. 

On teste plusieurs valeurs du paramètre de régularisation *C* afin d’identifier celle qui donne la meilleure performance. Pour chaque valeur, le modèle est entraîné sur les données d’apprentissage, puis évalué sur le jeu de validation à l’aide de l’**AUC**. 

Le solveur **liblinear** est choisi car il est plus rapide pour ce type de problème. 

Enfin, on conserve le modèle obtenant la meilleure AUC, qui sera ensuite utilisé pour les prédictions sur les jeux de validation et de test.

In [26]:
best_auc_rl = 0
best_C = 1.0
best_rl = None

for C in [3.0, 5.0, 10.0]:  
    rl = LogisticRegression(
        max_iter=400,           
        random_state=42,
        C=C,
        solver='liblinear',     
        penalty='l2'
    )
    rl.fit(X_train_scaled, y_train_split)
    y_val_pred = rl.predict_proba(X_val_scaled)[:, 1]
    auc = roc_auc_score(y_val_split, y_val_pred)
    print(f"C={C}: AUC={auc:.4f}")

    if auc > best_auc_rl:
        best_auc_rl = auc
        best_C = C
        best_rl = rl

print(f"\nMeilleur modèle pour la régression logistique: C={best_C}, AUC={best_auc_rl:.4f}")

y_val_pred_rl = best_rl.predict_proba(X_val_scaled)[:, 1]
y_test_pred_rl = best_rl.predict_proba(X_test_scaled)[:, 1]

C=3.0: AUC=0.7738
C=5.0: AUC=0.7739
C=10.0: AUC=0.7739

Meilleur modèle pour la régression logistique: C=10.0, AUC=0.7739


# Réseau de neurones

Pour le second modèle, on réalise un réseau de neurones. 

On teste plusieurs architectures (différentes tailles et nombres de couches cachées) afin d’identifier celle qui offre les meilleures performances. Encore une fois, chaque modèle est entraîné sur les données d’apprentissage, puis évalué sur le jeu de validation à l’aide de l’AUC. 

On utilise l’**early stopping** pour éviter le surapprentissage, ainsi qu’une **régularisation L2** pour améliorer la généralisation. 

L’architecture obtenant la meilleure AUC est finalement retenue pour effectuer les prédictions sur les jeux de validation et de test.

In [27]:
best_auc_rn = 0
architectures = [
    (150, 75),
    (200, 100, 50),
    (100, 50, 25),
    (150, 100, 50)
]

for hidden_layers in architectures:
    print(f"\nTest architecture: {hidden_layers}")
    rn = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.15,
        alpha=0.001,  
        learning_rate='adaptive',
        learning_rate_init=0.001,
        batch_size=256,
        verbose=False
    )
    rn.fit(X_train_scaled, y_train_split)
    y_val_pred = rn.predict_proba(X_val_scaled)[:, 1]
    auc = roc_auc_score(y_val_split, y_val_pred)
    print(f"AUC: {auc:.4f}")
    
    if auc > best_auc_rn:
        best_auc_rn = auc
        best_rn = rn
        best_arch = hidden_layers

print(f"\nMeilleure architecture pour le réseau de neurones: {best_arch}, AUC={best_auc_rn:.4f}")

y_val_pred_rn = best_rn.predict_proba(X_val_scaled)[:, 1]
y_test_pred_rn = best_rn.predict_proba(X_test_scaled)[:, 1]



Test architecture: (150, 75)
AUC: 0.7674

Test architecture: (200, 100, 50)
AUC: 0.7661

Test architecture: (100, 50, 25)
AUC: 0.7615

Test architecture: (150, 100, 50)
AUC: 0.7683

Meilleure architecture pour le réseau de neurones: (150, 100, 50), AUC=0.7683


# Mélange des deux modélisations

Afin d’améliorer les performances, on combine la régression logistique et le réseau de neurones à l’aide d’un ensemble pondéré. 

L’idée est de faire une moyenne pondérée des probabilités prédites par les deux modèles. On teste plusieurs valeurs du poids attribué au réseau de neurones, le reste étant attribué à la régression logistique, et on évalue chaque combinaison à l’aide de l’AUC sur le jeu de validation. 

Le poids donnant la meilleure AUC est retenu, puis utilisé pour générer les prédictions finales sur le jeu de test.

In [28]:
best_auc_melange = 0
best_weight = 0.5

for w_rn in np.arange(0.3, 0.8, 0.05):
    w_rl = 1 - w_rn
    y_val_pred_mel = w_rl * y_val_pred_rl + w_rn * y_val_pred_rn
    auc_mel = roc_auc_score(y_val_split, y_val_pred_mel)
    print(f"Poids RL={w_rl:.2f}, RN={w_rn:.2f}: AUC={auc_mel:.4f}")
    
    if auc_mel > best_auc_melange:
        best_auc_melange = auc_mel
        best_weight = w_rn

print(f"\nMeilleur poids: RL={1-best_weight:.2f}, RN={best_weight:.2f}")
print(f"Meilleur AUC mélange: {best_auc_melange:.4f}")

y_test_pred_melange = (1 - best_weight) * y_test_pred_rl + best_weight * y_test_pred_rn

Poids RL=0.70, RN=0.30: AUC=0.7766
Poids RL=0.65, RN=0.35: AUC=0.7767
Poids RL=0.60, RN=0.40: AUC=0.7766
Poids RL=0.55, RN=0.45: AUC=0.7765
Poids RL=0.50, RN=0.50: AUC=0.7762
Poids RL=0.45, RN=0.55: AUC=0.7759
Poids RL=0.40, RN=0.60: AUC=0.7755
Poids RL=0.35, RN=0.65: AUC=0.7749
Poids RL=0.30, RN=0.70: AUC=0.7743
Poids RL=0.25, RN=0.75: AUC=0.7736

Meilleur poids: RL=0.65, RN=0.35
Meilleur AUC mélange: 0.7767


# Récapitulatif des AUC

In [29]:
print(f"Régression logistique: {best_auc_rl:.4f}")
print(f"Réseau de neurones: {best_auc_rn:.4f}")
print(f"Mélange: {best_auc_melange:.4f}")

Régression logistique: 0.7739
Réseau de neurones: 0.7683
Mélange: 0.7767


# Création des fichiers de soumission

Enfin, on génére les fichiers de soumission. 

On utilise le fichier *sample_submission* fourni, puis on remplace la colonne **TARGET** par les probabilités prédites par chaque modèle (régression logistique, réseau de neurones et mélange). 

Trois fichiers *.csv* sont ainsi créés, ce qui permet de comparer les performances de chaque approche sur Kaggle. 

In [30]:
sub = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/sample_submission.csv', encoding='UTF-8')

# Fichier 1: Régression logistique
sub_rl = sub.copy()
sub_rl["TARGET"] = y_test_pred_rl
sub_rl.to_csv('submission_rl.csv', index=False)
print("submission_rl.csv créé")

# Fichier 2: Réseau de neurones
sub_rn = sub.copy()
sub_rn["TARGET"] = y_test_pred_rn
sub_rn.to_csv('submission_rn.csv', index=False)
print("submission_rn.csv créé")

# Fichier 3: Mélange
sub_melange = sub.copy()
sub_melange["TARGET"] = y_test_pred_melange
sub_melange.to_csv('submission_melange.csv', index=False)
print("submission_melange.csv créé")

submission_rl.csv créé
submission_rn.csv créé
submission_melange.csv créé


# Conclusion

En conclusion, les résultats obtenus sur Kaggle montrent que le modèle de réseau de neurones (AUC = 0.76585) performe légèrement mieux que la régression logistique (AUC = 0.76432), mais c’est l’ensemble pondéré des deux modèles qui donne les meilleures performances avec un AUC de 0.77020. 

Un AUC de 0.77020 signifie que le modèle a environ 77 % de chances de classer un client en défaut devant un client non défaillant, ce qui indique une bonne capacité de discrimination, même si le modèle n’est pas parfait.

Cela confirme que la combinaison de modèles complémentaires permet d’améliorer la capacité de prédiction par rapport à un modèle utilisé seul.
